# 03. Hyperparameter tuning and model comparison

In the previous notebook I built a baseline (TF-IDF on character 1-2 grams + logistic regression) and compared several manual variants with 5-fold CV. The best of them reached macro-F1 ≈ 0.887 (C=10). Here I tune the model more systematically and compare it with other model types.

All decisions are made with stratified 5-fold cross-validation on the train set only. The test set is used once, at the end, for the final model.

Plan:
1. Load the data and recreate the same train/test split and CV folds as before (so the results are comparable).
2. Tune logistic regression with Optuna (C, class_weight, ngram_range), maximizing mean macro-F1 in CV.
3. Compare the tuned model with the best manual variant: is the gain larger than the fold-to-fold std?
4. Try a decision tree and kNN (cosine distance) on the same TF-IDF features.
5. Choose the final model and evaluate it once on the test set.
6. Error analysis: read misclassified reviews and check whether the errors come from the model or from noisy labels.

In [36]:
import pandas as pd

df = pd.read_csv("../data/waimai_10k.csv") 
mask = df.duplicated(subset="review", keep=False)
df.loc[mask, "label"] = 1
df = df.drop_duplicates(subset="review").reset_index(drop=True)
df.shape  

(11980, 2)

In [37]:
from sklearn.model_selection import train_test_split
X = df["review"]
y = df['label']
X_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size =0.2,
    random_state=42,
    stratify=y
)

In [38]:
print(f"Train shape: {X_train.shape}, Test shape: {x_test.shape}")

Train shape: (9584,), Test shape: (2396,)


In [39]:
print(f'Train label distribution:\n{y_train.value_counts(normalize=True)}')
print(f'Test label distribution:\n{y_test.value_counts(normalize=True)}')

Train label distribution:
label
0    0.66611
1    0.33389
Name: proportion, dtype: float64
Test label distribution:
label
0    0.66611
1    0.33389
Name: proportion, dtype: float64


The data split is right.

In [40]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
def objective(trial):
    C = trial.suggest_float("C", 0.5, 100, log=True)
    cw = trial.suggest_categorical("class_weight", [None, "balanced"])
    ngram = trial.suggest_categorical("ngram", ["1-2", "1-3"])
    model = LogisticRegression(C=C, class_weight=cw, max_iter=1000)
    
    pipeline = Pipeline(steps=[
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, int(ngram.split("-")[1])))),
        ("lg", model)
    ])
    
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1_macro")
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

[I 2026-11-14 19:34:20,439] A new study created in memory with name: no-name-79746213-de8b-4ec1-9a47-6ca64fd0dbf8
[I 2026-11-14 19:34:27,187] Trial 0 finished with value: 0.8797105029717717 and parameters: {'C': 1.6280232848935992, 'class_weight': None, 'ngram': '1-3'}. Best is trial 0 with value: 0.8797105029717717.
[I 2026-11-14 19:34:30,515] Trial 1 finished with value: 0.8649704032337853 and parameters: {'C': 0.5384982983352221, 'class_weight': None, 'ngram': '1-2'}. Best is trial 0 with value: 0.8797105029717717.
[I 2026-11-14 19:34:33,688] Trial 2 finished with value: 0.8850710823611333 and parameters: {'C': 2.16195649441135, 'class_weight': None, 'ngram': '1-2'}. Best is trial 2 with value: 0.8850710823611333.
[I 2026-11-14 19:34:39,303] Trial 3 finished with value: 0.8868162003320867 and parameters: {'C': 43.96098442842632, 'class_weight': 'balanced', 'ngram': '1-3'}. Best is trial 3 with value: 0.8868162003320867.
[I 2026-11-14 19:34:46,051] Trial 4 finished with value: 0.8852

See the best params and the value.

In [41]:
print(study.best_params)
print(study.best_value)

{'C': 6.468601209051702, 'class_weight': 'balanced', 'ngram': '1-3'}
0.8902521106887924


In [42]:
df_trials = study.trials_dataframe().sort_values("value", ascending=False)
df_trials[["value", "params_C", "params_class_weight", "params_ngram"]].head(10)

,value,params_C,params_class_weight,params_ngram
19,0.890252,6.468601,balanced,1-3
16,0.889922,8.726076,balanced,1-3
21,0.889689,10.332638,balanced,1-3
24,0.889412,11.883149,balanced,1-3
14,0.889341,8.410681,balanced,1-3
15,0.889323,5.911997,balanced,1-3
17,0.888864,14.651223,balanced,1-3
22,0.888540,3.059264,balanced,1-3
25,0.888375,2.900099,balanced,1-2
11,0.888173,31.170345,balanced,1-3


I tuned logistic regression with Optuna (30 trials, maximizing macro-F1 in 5-fold CV on train over C, class_weight and n-gram range). The best configurations all use balanced class weights and 1-3 character n-grams, and C anywhere in ~4-20 gives almost the same score (CV macro-F1 ≈ 0.889-0.891, only ~0.004 above the manual C=10, which is within the fold-to-fold std).

In [43]:

from sklearn.metrics import make_scorer, fbeta_score, recall_score
from sklearn.metrics import precision_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f2_neg = make_scorer(fbeta_score, beta=2, pos_label=0)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000)),
])
scores = cross_validate(
    pipe, X_train, y_train, cv=cv,
    scoring={"f2_neg": f2_neg, "recall_neg": make_scorer(recall_score, pos_label=0)},
)

In [44]:
scoring = {
    "f2_neg": f2_neg,
    "recall_neg": make_scorer(recall_score, pos_label=0),
    "precision_neg": make_scorer(precision_score, pos_label=0),
    "f1_macro": "f1_macro",
}

In [45]:
def run_cv(model):
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    return {
        "macro_f1": s["test_f1_macro"].mean(),
        "macro_f1_std": s["test_f1_macro"].std(),
        "recall_0": s["test_recall_neg"].mean(),
        "precision_0": s["test_precision_neg"].mean(),
    }

In [46]:
pipe_lr = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, 3))),
    ("clf", LogisticRegression(C=10, class_weight="balanced", max_iter=1000)),
])
res_lr = run_cv(pipe_lr)
res_lr

{'macro_f1': np.float64(0.8894856015295733),
 'macro_f1_std': np.float64(0.008138277416381566),
 'recall_0': np.float64(0.9356190915718905),
 'precision_0': np.float64(0.919486638311972)}

### Final logistic regression

I fixed the final logistic regression configuration as TF-IDF on character 1-3 grams, C=10 and balanced class weights. In 5-fold CV on train it gets macro-F1 = 0.889 (std 0.008), with recall 0.936 and precision 0.919 for the negative class. This is only ~0.003 above the manual C=10 variant (0.887), which is within the fold-to-fold std, so I treat them as equally good. Compared with the dummy model (macro-F1 0.40) it is a large improvement, and its errors are balanced between missed complaints and false alarms.

In [47]:
from sklearn.tree import DecisionTreeClassifier

pipe_tree = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, 2))),
    ("clf", DecisionTreeClassifier(max_depth=30, class_weight="balanced", random_state=42)),
])
res_tree30 = run_cv(pipe_tree)
res_tree30

{'macro_f1': np.float64(0.8064831582012669),
 'macro_f1_std': np.float64(0.00478193985785355),
 'recall_0': np.float64(0.8572979136544066),
 'precision_0': np.float64(0.878485112299269)}

In [48]:
from sklearn.base import clone

res_tree = {}
for d in [10, 30, None]:
    m = clone(pipe_tree)
    m.set_params(clf__max_depth=d)
    res_tree[f"Tree depth={d}"] = run_cv(m)

pd.DataFrame(res_tree).T.round(4)

,macro_f1,macro_f1_std,recall_0,precision_0
Tree depth=10,0.8231,0.0086,0.8858,0.8796
Tree depth=30,0.8065,0.0048,0.8573,0.8785
Tree depth=None,0.8029,0.0092,0.8543,0.8761


### Decision tree

A decision tree on TF-IDF character n-grams is clearly worse than logistic regression: the best depth (10) gives macro-F1 = 0.823 vs 0.889. Deeper trees get worse (0.807 at depth 30, 0.803 without limit), which points to overfitting: the tree memorizes rare character combinations from the train set. With thousands of sparse features, a single tree splitting on one feature at a time is a poor fit for text, while a linear model can combine many weak signals.

In [49]:
from sklearn.neighbors import KNeighborsClassifier

res_knn = {}
for k in [5, 15, 30]:
    m = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, 2))),
        ("clf", KNeighborsClassifier(n_neighbors=k, metric="cosine", algorithm="brute")),
    ])
    res_knn[f"kNN k={k}"] = run_cv(m)

pd.DataFrame(res_knn).T.round(4)

,macro_f1,macro_f1_std,recall_0,precision_0
kNN k=5,0.8296,0.0133,0.8813,0.8893
kNN k=15,0.8425,0.0109,0.9030,0.8898
kNN k=30,0.8411,0.0141,0.9242,0.8765


### kNN

kNN with cosine distance on TF-IDF character 1-2 grams reaches macro-F1 = 0.843 at k=15, better than a decision tree (0.823) but clearly worse than logistic regression (0.889). Small k (5) is noisier (0.830), and larger k trades precision for recall of the negative class (recall 0.924, precision 0.877 at k=30) because the majority class dominates the neighbors. k=15 and k=30 are equally good within the fold-to-fold std.

In [50]:
def objective_tree(trial):
    depth = trial.suggest_int("max_depth", 3, 40)
    leaf = trial.suggest_int("min_samples_leaf", 1, 20)
    ngram = trial.suggest_categorical("ngram", ["1-2", "1-3"])
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, int(ngram.split("-")[1])))),
        ("clf", DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                                       class_weight="balanced", random_state=42)),
    ])
    return cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro").mean()

In [60]:
study_tree = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study_tree.optimize(objective_tree, n_trials=20)

print(study_tree.best_params)
print(study_tree.best_value)

[I 2026-11-14 20:08:51,108] A new study created in memory with name: no-name-10e260ce-8d10-453d-bd09-f6b9e7e3a09a
[I 2026-11-14 20:08:55,559] Trial 0 finished with value: 0.8022821828283584 and parameters: {'max_depth': 17, 'min_samples_leaf': 20, 'ngram': '1-2'}. Best is trial 0 with value: 0.8022821828283584.
[I 2026-11-14 20:09:01,292] Trial 1 finished with value: 0.8176811876500676 and parameters: {'max_depth': 8, 'min_samples_leaf': 4, 'ngram': '1-3'}. Best is trial 1 with value: 0.8176811876500676.
[I 2026-11-14 20:09:10,177] Trial 2 finished with value: 0.7957427521621632 and parameters: {'max_depth': 25, 'min_samples_leaf': 15, 'ngram': '1-3'}. Best is trial 1 with value: 0.8176811876500676.
[I 2026-11-14 20:09:22,399] Trial 3 finished with value: 0.790572446213055 and parameters: {'max_depth': 34, 'min_samples_leaf': 5, 'ngram': '1-3'}. Best is trial 1 with value: 0.8176811876500676.
[I 2026-11-14 20:09:26,198] Trial 4 finished with value: 0.8033105120912547 and parameters: {'

{'max_depth': 11, 'min_samples_leaf': 1, 'ngram': '1-2'}
0.8195540466782261


In [ ]:

print(study_tree.best_params)
print(study_tree.best_value)

{'max_depth': 11, 'min_samples_leaf': 1, 'ngram': '1-2'}
0.8195540466782261


### Decision tree: Optuna

Tuning the tree with Optuna (20 trials over max_depth, min_samples_leaf and n-gram range) gave macro-F1 = 0.820 (max_depth=11, min_samples_leaf=1, 1-2 grams), the same as the manual search (0.823 at depth 10) within the fold-to-fold std. So the tree plateaus around 0.82 and stays far below logistic regression (0.889).

In [52]:
def objective_knn(trial):
    k = trial.suggest_int("n_neighbors", 5, 30)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, 2))),
        ("clf", KNeighborsClassifier(n_neighbors=k, weights=weights,
                                     metric="cosine", algorithm="brute")),
    ])
    return cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro").mean()

study_knn = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_knn.optimize(objective_knn, n_trials=12)

print(study_knn.best_params)
print(study_knn.best_value)

[I 2026-11-14 19:56:46,201] A new study created in memory with name: no-name-f0d587af-8e5b-42db-9cb3-43929c0982ad
[I 2026-11-14 19:56:51,965] Trial 0 finished with value: 0.8415252901009931 and parameters: {'n_neighbors': 14, 'weights': 'uniform'}. Best is trial 0 with value: 0.8415252901009931.
[I 2026-11-14 19:56:55,912] Trial 1 finished with value: 0.8454786961666864 and parameters: {'n_neighbors': 20, 'weights': 'uniform'}. Best is trial 1 with value: 0.8454786961666864.
[I 2026-11-14 19:56:59,928] Trial 2 finished with value: 0.8329593042518484 and parameters: {'n_neighbors': 6, 'weights': 'uniform'}. Best is trial 1 with value: 0.8454786961666864.
[I 2026-11-14 19:57:04,779] Trial 3 finished with value: 0.8453085996313368 and parameters: {'n_neighbors': 23, 'weights': 'distance'}. Best is trial 1 with value: 0.8454786961666864.
[I 2026-11-14 19:57:08,440] Trial 4 finished with value: 0.844066183118013 and parameters: {'n_neighbors': 26, 'weights': 'uniform'}. Best is trial 1 with

{'n_neighbors': 30, 'weights': 'distance'}
0.8457785940708591


### kNN: Optuna

Tuning kNN with Optuna (12 trials over n_neighbors in 5-30 and weights, cosine distance, 1-2 grams) gave macro-F1 = 0.846 (k=30, distance weighting). This is the same as the manual search (0.843 at k=15) within the fold-to-fold std. The best k is at the upper edge of the searched range, but manual results suggest a plateau. kNN stays well below logistic regression (0.889).

In [61]:
best = study_tree.best_params   # {'max_depth': 11, 'min_samples_leaf': 1, 'ngram': '1-2'}

pipe_tr = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, int(best["ngram"].split("-")[1])))),
    ("clf", DecisionTreeClassifier(
        max_depth=best["max_depth"],
        min_samples_leaf=best["min_samples_leaf"],
        class_weight="balanced",
        random_state=42,
    )),
])
res_tr = run_cv(pipe_tr)


In [55]:
pipe_kn = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, 2))),
    ("clf", KNeighborsClassifier(**study_knn.best_params, metric="cosine", algorithm="brute")),
])
res_kn = run_cv(pipe_kn)

In [66]:
final_table = pd.DataFrame({
    "LogReg C=10, balanced, 1-3 grams": res_lr,
    "kNN k=30, distance (Optuna)": res_kn,
    "Tree depth=11 (Optuna)": res_tr,
}).T.round(4).sort_values("macro_f1", ascending=False)
print(final_table)

                                  macro_f1  macro_f1_std  recall_0  \
LogReg C=10, balanced, 1-3 grams    0.8895        0.0081    0.9356   
kNN k=30, distance (Optuna)         0.8458        0.0136    0.9189   
Tree depth=11 (Optuna)              0.8196        0.0079    0.8866   

                                  precision_0  
LogReg C=10, balanced, 1-3 grams       0.9195  
kNN k=30, distance (Optuna)            0.8840  
Tree depth=11 (Optuna)                 0.8755  


## Model comparison and final choice

Criterion (fixed before looking at the test set): the final model is the one with the highest CV macro-F1; if differences are within the fold-to-fold std, the simpler/faster model is preferred.

| Model | macro-F1 | std | recall (class 0) | precision (class 0) |
|---|---|---|---|---|
| Logistic regression (C=10, balanced, 1-3 grams) | 0.8895 | 0.0081 | 0.9356 | 0.9195 |
| kNN (k=30, distance weights, Optuna) | 0.8458 | 0.0136 | 0.9189 | 0.8840 |
| Decision tree (depth 11, Optuna) | 0.8196 | 0.0079 | 0.8866 | 0.8755 |

Logistic regression is clearly the best: its advantage over kNN (~0.044) and the decision tree (~0.070) is much larger than the CV std (0.008-0.014), and it has the most balanced precision and recall for the negative class. All three model families were tuned with a comparable search (Optuna or a manual grid), so the gap is not caused by tuning effort. Final model: logistic regression (TF-IDF on character 1-3 grams, C=10, balanced class weights).